# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Cite as: {metadata.cite_as if hasattr(metadata, 'cite_as') else getattr(metadata, 'citeAs', None)}")
print(f"Published: {metadata.date_published}")
print(f"Fields: {getattr(metadata, 'keywords', [])}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets, their @id and description

print("Available Record Sets:\n---------------------")
for rs in dataset.record_sets:
    print(f"@id: {rs.id}\n  Name: {rs.name}\n  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.id} (name: {field.name}, dataType: {field.data_type})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets
record_sets_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Show columns for the first record set (as an example)
if record_sets_ids:
    first_record_set_id = record_sets_ids[0]
    print(f"Columns in Record Set {first_record_set_id}: ", dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, work with the first record set and its numeric columns
import numpy as np

# Identify numeric columns (fields of type Float or Integer)
record_set = dataset.record_sets[0]
df = dataframes[record_set.id]
numeric_fields = [field.id for field in record_set.fields if field.data_type in ["Float", "Integer", "Number"] and field.id in df.columns]

if numeric_fields:
    numeric_field = numeric_fields[0]
    print(f"Using numeric field: {numeric_field}")
    
    # Handle missing data and convert column to numeric if needed
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    
    # Filtering
    threshold = df[numeric_field].mean() if df[numeric_field].notnull().sum() else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())
    
    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
        (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() else 1)
    )
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Attempt grouping by a likely categorical field (e.g. a field containing 'Type', 'Sample', or similar)
    group_field_candidates = [field.id for field in record_set.fields if (field.data_type == "Text" or field.data_type == "String") and field.id != numeric_field]
    if group_field_candidates:
        group_field = group_field_candidates[0]
        if group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped data by {group_field}:\n")
            print(grouped_df.head())
else:
    print("No numeric fields found for this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in record set {record_set.id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    # If group_field available, boxplot
    if group_field_candidates and group_field_candidates[0] in df.columns:
        group_field = group_field_candidates[0]
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this exploration, we loaded and explored the FAIR^2 dataset using `mlcroissant`.

- The dataset provides rich information on protein abundance and modifications from mass spectrometry analysis of human mast cell extracellular vesicles.
- We demonstrated programmatic loading, inspection, and DataFrame construction directly from Croissant record sets using their `@id` fields.
- We performed basic filtering and normalization of numeric fields, and showed how to group data by categorical variables for summarization.
- Simple visualizations illustrated field distributions and relationships.

**Next steps:** You can extend this notebook to do statistical analysis, machine learning, or domain-specific biomarker identification, leveraging the rich structure provided in this Croissant dataset.